# **Ongoing fire tracker in Borneo, Indonesia**
This tool was developed to support the work of Orangutan Foundation International Canada. It is a live, auto refreshing map which tracks fire activity in Borneo, Indonesia, the lone remaining habitat of the critically endangered bornean orangutan. 

## **Data sources**
### Fire data
Data on active fires was sourced from NASA's Fire Information for Resource Management System (FIRMS), specifically the Visible Infrared Imaging Radiometer Suite (VIIRS) aboard the NOAA-20 and NOAA-21 satellites, at a spatial resolution of 375m. This instrument detects active fires and thermal anomalies, made available within approximately 3 hours of satellite observation (Near Real-Time, or NRT, product). The data is available at https://firms.modaps.eosdis.nasa.gov/map/#d:24hrs;@0.0,0.0,3.0z. 

### Bornean orangutan range
Spatial data on bornean orangutan range was sourced from the IUCN's Red List of Threatened Species (https://www.iucnredlist.org/resources/spatial-data-download), using the 2024 range polygon release, and filtered.

### Protected areas
Spatial data on protected areas was retrieved from Protected Planet's World Database on Protected Areas (WDPA), maintained by UNEP-WCMC and IUCN (https://www.protectedplanet.net/en/thematic-areas/wdpa?tab=WDPA), using the 2026 release, and filtered to Indonesia. Protected area boundaries were pre-bounded to the island of Borneo prior to analysis, a one-time preprocessing step separate from the pipeline below, and exported as a shapefile for analysis. The protected areas within ape ranges file is 'Indo_PAs_borneo.shp.'

In [4]:
import pandas as pd
import geopandas as gpd

MAP_KEY = '2790d08a201a95961b6d0d86c5131561'
AREA = '108.5,-4.5,119.5,7.5'   # Borneo bounding box: west,south,east,north
DAY_RANGE = 5

sources = ['VIIRS_NOAA20_NRT', 'VIIRS_NOAA21_NRT']
all_fires = []
for source in sources:
    url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{source}/{AREA}/{DAY_RANGE}'
    chunk = pd.read_csv(url)
    print(f"{source}: {len(chunk)} detections")
    all_fires.append(chunk)

fire_df = pd.concat(all_fires, ignore_index=True).drop_duplicates()
print(f"Combined total (after dedup): {len(fire_df)}")

fire_gdf = gpd.GeoDataFrame(
    fire_df,
    geometry=gpd.points_from_xy(fire_df.longitude, fire_df.latitude),
    crs='EPSG:4326'
)

VIIRS_NOAA20_NRT: 28351 detections
VIIRS_NOAA21_NRT: 29543 detections
Combined total (after dedup): 57894


### Filter for high confidence

In [5]:
print(fire_gdf['confidence'].value_counts())
fire_gdf = fire_gdf[fire_gdf['confidence'].isin(['n', 'h'])]
print(f"{len(fire_gdf)} detections after filtering low-confidence")


confidence
n    52883
l     2618
h     2393
Name: count, dtype: int64
55276 detections after filtering low-confidence


In [6]:
ape_ranges = gpd.read_file(r'C:\Users\OMILLER1\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\data\ape_ranges.shp')
orangutan_range = ape_ranges[
    (ape_ranges['sci_name'] == 'Pongo pygmaeus') &
    (ape_ranges['presence'] == 1)
]

fires_in_range = gpd.sjoin(fire_gdf, orangutan_range, predicate='within')
print(f"{len(fires_in_range)} fire detections within Bornean orangutan range")

11831 fire detections within Bornean orangutan range


In [8]:
Indo_PAs = gpd.read_file(r'C:\Users\OMILLER1\OneDrive - United Nations\Desktop\Career\Borneo Fire Tracker\data\Indo_PAs_borneo.shp')

In [9]:
fires_proj = fires_in_range.to_crs('EPSG:6933')
fires_proj['geometry'] = fires_proj.geometry.buffer(187.5)
burned_area = fires_proj.dissolve().geometry.area.sum() / 10_000
print(f"Approx. {burned_area:,.0f} hectares of orangutan range affected in the last 5 days")

Approx. 72,967 hectares of orangutan range affected in the last 5 days


In [ ]:
range_area_ha = orangutan_range.dissolve().to_crs('EPSG:6933').geometry.area.sum() / 10_000
pct_affected = 41301/ range_area_ha * 100
print(f"That's {pct_affected:.2f}% of total confirmed range ({range_area_ha:,.0f} ha)")

That's 0.52% of total confirmed range (14,002,766 ha)


In [15]:
import folium
from folium import GeoJson
import datetime
import numpy as np

m_fire_sized = folium.Map(location=[-1, 112], zoom_start=7, tiles=None)

folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/rastertiles/light_all/{z}/{x}/{y}.png?key=cb1_28oy_1_05571f4015c3314d60d78268',
    attr='&copy; OpenStreetMap, &copy; CARTO',
    subdomains='abcd', max_zoom=20, control=False
).add_to(m_fire_sized)

# Custom panes to lock stacking order, fires always render above range/PA layers,
# regardless of add order or LayerControl toggling
folium.map.CustomPane('pa_pane', z_index=390).add_to(m_fire_sized)
folium.map.CustomPane('range_pane', z_index=395).add_to(m_fire_sized)
folium.map.CustomPane('fire_pane', z_index=650).add_to(m_fire_sized)

from folium.plugins import Fullscreen

Fullscreen(
    position='topleft',
    title='Full-screen',
    title_cancel='Exit full-screen',
    force_separate_button=True
).add_to(m_fire_sized)

orangutan_range = ape_ranges[
    (ape_ranges['sci_name'] == 'Pongo pygmaeus') &
    (ape_ranges['presence'] == 1)
]
orangutan_range_dissolved = orangutan_range.dissolve()
folium.GeoJson(
    orangutan_range_dissolved,
    style_function=lambda x: {
        'fillColor': '#006400',
        'color': '#006400',
        'weight': 0.2,
        'fillOpacity': 0.2
    },
    name='Bornean orangutan range',
    pane='range_pane'
).add_to(m_fire_sized)

folium.GeoJson(
    Indo_PAs,
    style_function=lambda x: {
        'fillColor': '#00509d',
        'color': '#00509d',
        'weight': 0.6,
        'fillOpacity': 0.1
    },
    name='Protected areas',
    pane='pa_pane'
).add_to(m_fire_sized)

today = fires_in_range['acq_date'].max()
today_dt = datetime.datetime.strptime(today, '%Y-%m-%d')

def recency_color(acq_date):
    days_ago = (today_dt - datetime.datetime.strptime(acq_date, '%Y-%m-%d')).days
    if days_ago == 0:
        return '#7f0000'
    elif days_ago <= 1:
        return '#c1121f'
    elif days_ago <= 2:
        return '#e5533c'
    else:
        return '#f4a582'

frp_max = fires_in_range['frp'].max()

def scale_radius_log(frp, min_r=2, max_r=8):
    log_frp = np.log1p(frp)
    log_max = np.log1p(frp_max)
    return min_r + (log_frp / log_max) * (max_r - min_r)

for row in fires_in_range.itertuples():
    popup_html = f"""
    <div style="font-family: Roboto, sans-serif; font-size: 12px;">
        <b>Fire Detection</b><br>
        <b>Date:</b> {row.acq_date}<br>
        <b>FRP:</b> {row.frp:.1f} MW<br>
        <b>Coordinates:</b> {row.latitude:.2f}, {row.longitude:.2f}
    </div>
    """
    folium.CircleMarker(
        [row.latitude, row.longitude],
        radius=scale_radius_log(row.frp),
        color=recency_color(row.acq_date),
        fill=True,
        fill_opacity=0.8,
        weight=0,
        popup=folium.Popup(popup_html, max_width=220),
        pane='fire_pane'
    ).add_to(m_fire_sized)

folium.LayerControl(collapsed=False).add_to(m_fire_sized)

m_fire_sized.get_root().html.add_child(folium.Element("""
<style>
.leaflet-control-layers {
    top: 15px !important;
    right: 15px !important;
    padding: 10px 14px !important;
    font-size: 13px !important;
    border-radius: 6px !important;
}
.leaflet-control-layers-overlays label {
    margin-bottom: 4px !important;
    display: flex !important;
    align-items: center !important;
}
.leaflet-control-layers input[type="checkbox"] {
    transform: scale(1.1);
    margin-right: 4px !important;
}
.leaflet-control-layers-list::before {
    content: 'Layers';
    font-weight: bold;
    font-size: 13px;
    display: block;
    margin-bottom: 6px;
    padding-bottom: 4px;
    border-bottom: 1px solid #eeeeee;
}
.leaflet-control-layers-overlays label:nth-of-type(1)::after {
    content: '';
    display: inline-block;
    width: 12px;
    height: 12px;
    background-color: #006400;
    border-radius: 2px;
    margin-left: 6px;
    vertical-align: middle;
}
.leaflet-control-layers-overlays label:nth-of-type(2)::after {
    content: '';
    display: inline-block;
    width: 12px;
    height: 12px;
    background-color: #00509d;
    border-radius: 2px;
    margin-left: 6px;
    vertical-align: middle;
}
</style>
"""))

legend_frp_values = [3, 15, 100]  # MW

low_r = scale_radius_log(3)
mid_r = scale_radius_log(15)
high_r = scale_radius_log(100)

legend_js = f'''
<script>
window.addEventListener('load', function() {{
    var legend = L.control({{position: 'bottomleft'}});
    legend.onAdd = function (map) {{
        var div = L.DomUtil.create('div', 'info legend');
        div.style.background = 'white';
        div.style.padding = '10px 14px';
        div.style.borderRadius = '6px';
        div.style.fontFamily = 'Roboto';
        div.style.boxShadow = '0 1px 4px rgba(0,0,0,0.3)';
        div.innerHTML = `
            <div style="font-size: 12px; margin-bottom: 4px;"><b>Recency</b></div>
            <div style="width: 130px; height: 8px; border-radius: 4px; margin-bottom: 3px;
                 background: linear-gradient(to right, #7f0000, #c1121f, #e5533c, #f4a582);"></div>
            <div style="display: flex; justify-content: space-between; font-size: 9px; width: 130px; margin-bottom: 8px;">
                 <span>Today</span><span>5 days ago</span>
            </div>
            <div style="font-size: 12px; margin-bottom: 4px;"><b>Fire intensity (FRP, MW)</b></div>
            <div style="display: flex; align-items: center; gap: 8px;">
                 <div style="display: flex; align-items: center;">
                     <div style="width: {low_r*1.2}px; height: {low_r*1.2}px; border-radius: 50%; background-color: #c1121f;"></div>
                     <span style="font-size: 9px; margin-left: 3px;">3</span>
                 </div>
                 <div style="display: flex; align-items: center;">
                     <div style="width: {mid_r*1.2}px; height: {mid_r*1.2}px; border-radius: 50%; background-color: #c1121f;"></div>
                     <span style="font-size: 9px; margin-left: 3px;">15</span>
                 </div>
                 <div style="display: flex; align-items: center;">
                     <div style="width: {high_r*1.2}px; height: {high_r*1.2}px; border-radius: 50%; background-color: #c1121f;"></div>
                     <span style="font-size: 9px; margin-left: 3px;">100</span>
                 </div>
            </div>
        `;
        return div;
    }};
    legend.addTo({m_fire_sized.get_name()});
}});
</script>
'''
m_fire_sized.get_root().html.add_child(folium.Element(legend_js))

m_fire_sized.save('borneo_fire_map_sized.html')

import webbrowser
webbrowser.open('borneo_fire_map_sized.html')

True

In [75]:
print(orangutan_range.columns.tolist())
print(orangutan_range[['sci_name']].shape)  # how many rows/polygons
# if there's a presence/status column:
print(orangutan_range['presence'].value_counts()) 

['id_no', 'sci_name', 'presence', 'origin', 'seasonal', 'compiler', 'yrcompiled', 'citation', 'subspecies', 'subpop', 'source', 'island', 'tax_comm', 'dist_comm', 'generalisd', 'legend', 'kingdom', 'phylum', 'class', 'order_', 'family', 'genus', 'category', 'marine', 'terrestria', 'freshwater', 'SHAPE_Leng', 'SHAPE_Area', 'geometry']
(4, 1)
presence
1    4
Name: count, dtype: int64


In [56]:
print(fires_in_range['frp'].describe())

count    4947.000000
mean        7.261898
std        12.742513
min         0.320000
25%         1.765000
50%         3.380000
75%         7.230000
max       221.350000
Name: frp, dtype: float64


In [80]:
print(f"Total PA records: {len(Indo_PAs)}")
print(Indo_PAs.crs)

Total PA records: 969
EPSG:4326


In [55]:
orangutan_range = ape_ranges[
    (ape_ranges['sci_name'] == 'Pongo pygmaeus') &
    (ape_ranges['presence'] == 1)
]

In [62]:
orangutan_range_dissolved = orangutan_range.dissolve()
range_area_ha = orangutan_range_dissolved.to_crs('EPSG:6933').geometry.area.sum() / 10_000
print(f"Total Bornean orangutan range (extant only): {range_area_ha:,.0f} hectares")

Total Bornean orangutan range (extant only): 14,002,766 hectares


In [ ]:
import pandas as pd
import geopandas as gpd
import time

MAP_KEY = '2790d08a201a95961b6d0d86c5131561'
SOURCE = 'VIIRS_NOAA20_NRT'
AREA = '108.5,-4.5,119.5,7.5'

# Generate 5-day chunks from season start to today
date_starts = pd.date_range(start='2026-07-01', end='2026-09-17', freq='5D').strftime('%Y-%m-%d')

all_fires = []
for date in date_starts:
    url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SOURCE}/{AREA}/5/{date}'
    chunk = pd.read_csv(url)
    all_fires.append(chunk)
    print(f"{date}: {len(chunk)} detections")
    time.sleep(1)

fire_df_season = pd.concat(all_fires, ignore_index=True).drop_duplicates()
print(f"\nTotal detections since July 1: {len(fire_df_season)}")

# Build GeoDataFrame
fire_gdf_season = gpd.GeoDataFrame(
    fire_df_season,
    geometry=gpd.points_from_xy(fire_df_season.longitude, fire_df_season.latitude),
    crs='EPSG:4326'
)

# Filter to reasonable confidence
fire_gdf_season = fire_gdf_season[fire_gdf_season['confidence'].isin(['n', 'h'])]

# Spatial join against confirmed-extant orangutan range (presence == 1)
fires_in_range_season = gpd.sjoin(fire_gdf_season, orangutan_range, predicate='within')
print(f"{len(fires_in_range_season)} detections within confirmed orangutan range since July 1")

# Buffer + dissolve to estimate affected area
fires_proj_season = fires_in_range_season.to_crs('EPSG:6933')
fires_proj_season['geometry'] = fires_proj_season.geometry.buffer(187.5)
burned_area_season = fires_proj_season.dissolve().geometry.area.sum() / 10_000

# Compare against total range
range_area_ha = orangutan_range.dissolve().to_crs('EPSG:6933').geometry.area.sum() / 10_000
pct_affected_season = burned_area_season / range_area_ha * 100

print(f"\nSince July 1, 2026:")
print(f"Approx. {burned_area_season:,.0f} hectares of confirmed orangutan range affected")
print(f"That's {pct_affected_season:.1f}% of total confirmed range ({range_area_ha:,.0f} ha)")

2026-07-01: 429 detections
2026-07-06: 1045 detections
2026-07-11: 1803 detections
2026-07-16: 1532 detections
2026-07-21: 2413 detections
2026-07-26: 1716 detections
2026-07-31: 5034 detections
2026-08-05: 11115 detections
2026-08-10: 9023 detections
2026-08-15: 16339 detections
2026-08-20: 13718 detections
2026-08-25: 27650 detections
2026-08-30: 27410 detections
2026-09-04: 14624 detections
2026-09-09: 19315 detections
2026-09-14: 11131 detections

Total detections since July 1: 164297
23614 detections within confirmed orangutan range since July 1

Since July 1, 2026:
Approx. 149,401 hectares of confirmed orangutan range affected
That's 1.1% of total confirmed range (14,002,766 ha)


In [ ]:
import os
print(os.path.abspath('borneo_fire_map_sized.html'))

c:\Users\OMILLER1\OneDrive - United Nations\Desktop\Career\Borneo Fire Tracker\borneo_fire_map_sized.html


VIIRS_NOAA20_NRT: 28351 detections
VIIRS_NOAA21_NRT: 29543 detections

Combined total (after dedup): 57894
